# Session 4: Generalization & Robustness Assessment

A model that performs well on average may fail for specific patient populations or when data quality degrades. This session evaluates whether our risk classifier generalizes across demographic subgroups and degrades gracefully under real-world conditions.

---

## What You'll Do Today

**Part 1: Generalization & Fairness**
- Evaluate model accuracy across age, gender demographics
- Calculate TPR, FPR, PPV disparities between groups
- Identify and document performance gaps

**Part 2: Cross-Validation Stability**
- 5-fold GroupKFold cross-validation
- Assess model stability across different patient splits

**Part 3: Robustness Testing**
- Feature category analysis and masking tests
- Random feature loss degradation curves
- Single feature removal impact

**Part 4: Error Analysis**
- Outlier identification and performance comparison
- False Negative and False Positive deep dives
- Risk score distributions for error cases

**Part 5: Combined Assessment Report**
- Full robustness report with deployment recommendations

---

## Why This Matters

Healthcare AI systems have historically shown bias and brittleness:

| Requirement | Why It Matters |
|-------------|----------------|
| **EU AI Act Article 10** | High-risk AI must be tested for bias across protected groups |
| **GDPR Article 22** | Automated decisions must not discriminate |
| **SwissCare Policy** | Model performance must be documented for each demographic subgroup |
| **Graceful Degradation** | System must maintain >90% of baseline when feature categories are missing |

 **SwissCare Policy**: *"Model performance must be documented for each demographic subgroup before deployment. Disparities >5% in sensitivity require remediation plan."*

---

## Setup & Imports

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src/ xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import pickle
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix

print("Libraries imported")


In [ ]:
import os

DATA_DIR = os.path.join(REPO_PATH, 'data', 'week_2')
RAW_DATA_DIR = os.path.join(REPO_PATH, 'data', 'week_1', 'processed_data', 'csv')
OUTPUT_DIR = os.path.join(REPO_PATH, 'data', 'week_3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"DATA_DIR:     {DATA_DIR}")
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

---

## Load Data, Train/Test Split & Load Pre-Trained Model

We load classifier training data and patient demographics, then load the tuned XGBoost model from Session 2 that will be the subject of all subsequent analysis.


In [ ]:
# Load classifier data
classifier_data = pd.read_csv(os.path.join(DATA_DIR, 'classifier_training_data_22_features.csv'), low_memory=False)

label_col = 'will_have_high_risk_event_next_30d'
patient_col = 'patient_id'

# Load patient demographics
patients_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'patients.csv'))

print(f"Classifier data: {len(classifier_data):,} instances")
print(f"Patients data: {len(patients_df):,} patients")

# Convert bmi_category from string to numeric (guard for already-numeric data)
if 'bmi_category' in classifier_data.columns and classifier_data['bmi_category'].dtype == 'object':
    bmi_mapping = {'underweight': 1, 'normal': 2, 'overweight': 3, 'obese': 4}
    classifier_data['bmi_category'] = classifier_data['bmi_category'].map(bmi_mapping)

# Drop rows with NaN labels
valid_mask = classifier_data[label_col].notna()
classifier_data = classifier_data[valid_mask].copy()
classifier_data[label_col] = classifier_data[label_col].astype(int)

In [ ]:
# Merge demographics
patient_id_col = None
for col in ['Id', 'id', 'ID', 'PATIENT', 'patient_id']:
    if col in patients_df.columns:
        patient_id_col = col
        break

demo_cols = [patient_id_col]
for col in ['BIRTHDATE', 'birthdate', 'GENDER', 'gender', 'RACE', 'race']:
    if col in patients_df.columns:
        demo_cols.append(col)

demographics = patients_df[demo_cols].copy()
demographics = demographics.rename(columns={patient_id_col: patient_col})
demographics.columns = [c.lower() for c in demographics.columns]

data_with_demo = classifier_data.merge(demographics, on=patient_col, how='left')

print(f"Merged data shape: {data_with_demo.shape}")
gender_pct = data_with_demo['gender'].notna().mean()
print(f"Patients with demographics: {gender_pct:.1%}")

In [ ]:
# Create age groups from the per-instance age_at_date feature (already in the dataset),
# NOT from birthdate + a single reference date.  age_at_date gives the patient's age
# at the time of each observation, which is consistent with how the model sees the data.
if 'age_at_date' in data_with_demo.columns:
    data_with_demo['age'] = data_with_demo['age_at_date']
    data_with_demo['age_group'] = pd.cut(
        data_with_demo['age'],
        bins=[18, 40, 55, 70, 85, 150],
        labels=['18-40', '41-55', '56-70', '71-85', '85+'],
        include_lowest=True
    )
    print(f"Age source: age_at_date (per-instance)")
    print(f"Age range: {data_with_demo['age'].min():.0f} - {data_with_demo['age'].max():.0f} years")
    print(f"\nAge Group Distribution:")
    print(data_with_demo['age_group'].value_counts().sort_index())
elif 'birthdate' in data_with_demo.columns:
    # Fallback: compute from birthdate and per-instance date
    data_with_demo['birthdate'] = pd.to_datetime(data_with_demo['birthdate'])
    instance_dates = pd.to_datetime(data_with_demo['date'])
    data_with_demo['age'] = ((instance_dates - data_with_demo['birthdate']).dt.days // 365).astype(float)
    data_with_demo['age_group'] = pd.cut(
        data_with_demo['age'],
        bins=[18, 40, 55, 70, 85, 150],
        labels=['18-40', '41-55', '56-70', '71-85', '85+'],
        include_lowest=True
    )
    print(f"Age source: birthdate - instance date (per-instance)")
    print(f"Age range: {data_with_demo['age'].min():.0f} - {data_with_demo['age'].max():.0f} years")
    print(f"\nAge Group Distribution:")
    print(data_with_demo['age_group'].value_counts().sort_index())
else:
    print("WARNING: No age information available")

In [ ]:
# Prepare features (exclude demographics from features)
# Use exact-match for 'age' and 'age_group' to keep 'age_at_date' as a feature
exclude_cols = ['patient_id', 'date', 'age', 'age_group']
exclude_patterns = ['label', 'risk', 'target', 'will_have', 'next',
                    'birth', 'gender', 'race', 'survival']

feature_cols = [col for col in data_with_demo.columns
                if col not in exclude_cols
                and not any(pat in col.lower() for pat in exclude_patterns)]

# NaN passthrough — this notebook uses only XGBoost, which handles NaN natively.
# fillna(0) would create clinically impossible values (47.8% of eGFR = 0, 9.9% of HbA1c = 0).
X = data_with_demo[feature_cols]
y = data_with_demo[label_col]
groups = data_with_demo[patient_col]

print(f"Features: {len(feature_cols)}")
print(f"Feature list: {feature_cols}")
print(f"Positive rate: {y.mean():.2%}")
print(f"NaN passthrough: XGBoost handles missing values natively")


In [ ]:
# Patient-level train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_data = data_with_demo.iloc[test_idx].copy()

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


In [ ]:
# Load pre-trained XGBoost model from Session 2
model_path = os.path.join(OUTPUT_DIR, 'best_xgboost_model.pkl')

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"{model_path} not found. Run Session 2 first to train and save the model."
    )

try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
except (EOFError, pickle.UnpicklingError) as e:
    raise RuntimeError(
        f"Model file is corrupted ({e}). Re-run Session 2's save cell to regenerate it."
    )

print(f"Loaded model from: {model_path}")
print(f"Model parameters: max_depth={model.max_depth}, "
      f"learning_rate={model.learning_rate}, "
      f"n_estimators={model.n_estimators}")

y_prob = model.predict_proba(X_test)[:, 1]

baseline_auroc = roc_auc_score(y_test, y_prob)
baseline_prauc = average_precision_score(y_test, y_prob)

print(f"\nBASELINE PERFORMANCE")
print(f"  AUROC: {baseline_auroc:.4f}")
print(f"  PR-AUC: {baseline_prauc:.4f}")

---

# Part 1: Generalization & Fairness

We evaluate the same model on different patient subgroups. The key question: **Does the model perform equally well for all groups?**

---

<details>
<summary><strong>Hint 1 — Subgroup metrics approach</strong> (click to expand)</summary>

For each unique group value, create a boolean mask, subset `y_test` and `y_prob`, then call `roc_auc_score()` and `average_precision_score()`. Skip groups with too few samples.
</details>

In [ ]:
def calculate_subgroup_metrics(test_data, y_test, y_prob, group_col, min_samples=20, min_events=5):
    """
    Calculate performance metrics for each subgroup.

    Args:
        test_data: DataFrame with group column
        y_test: true labels
        y_prob: predicted probabilities
        group_col: column to group by (e.g., 'age_group', 'gender')
        min_samples: minimum samples needed for reliable metrics
        min_events: minimum positive cases needed for AUROC

    Returns:
        pd.DataFrame with columns: group, n, n_events, auroc, prauc
    """
    results = []

    # TODO: Loop through unique values in group_col
    #   For each group:
    #     1. Create a boolean mask: test_data[group_col] == group
    #     2. Get y_true_sub and y_prob_sub using the mask
    #     3. Skip if fewer than min_samples or min_events
    #     4. Compute roc_auc_score and average_precision_score
    #     5. Append dict with 'group', 'n', 'n_events', 'auroc', 'prauc'
    for group in test_data[group_col].dropna().unique():
        mask = test_data[group_col] == group
        y_true_sub = y_test[mask]
        y_prob_sub = y_prob[mask]

        if len(y_true_sub) < min_samples or y_true_sub.sum() < min_events:
            continue

        # TODO: Compute metrics
        auroc = None
        prauc = None

        results.append({
            'group': group,
            'n': len(y_true_sub),
            'n_events': int(y_true_sub.sum()),
            'auroc': auroc,
            'prauc': prauc,
        })

    return pd.DataFrame(results) if results else pd.DataFrame()

print("Subgroup metrics function defined")

### Performance by Age Group

In [ ]:
if 'age_group' in test_data.columns:
    age_metrics = calculate_subgroup_metrics(test_data, y_test, y_prob, 'age_group')


    print("Performance by Age Group:")
    print("=" * 80)
    print(age_metrics.to_string(index=False))
else:
    age_metrics = pd.DataFrame()

In [ ]:
if len(age_metrics) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # AUROC by age group
    ax = axes[0]
    bars = ax.bar(age_metrics['group'].astype(str), age_metrics['auroc'], color='steelblue', alpha=0.8)
    ax.axhline(y=baseline_auroc, color='red', linestyle='--', linewidth=2, label=f'Overall ({baseline_auroc:.3f})')
    ax.set_xlabel('Age Group')
    ax.set_ylabel('AUROC')
    ax.set_title('Model Discrimination by Age Group')
    ax.legend()
    ax.set_ylim([0.5, 1.0])
    for bar, val in zip(bars, age_metrics['auroc']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

    # PR-AUC by age group
    ax = axes[1]
    bars = ax.bar(age_metrics['group'].astype(str), age_metrics['prauc'], color='darkorange', alpha=0.8)
    ax.axhline(y=baseline_prauc, color='red', linestyle='--', linewidth=2, label=f'Overall ({baseline_prauc:.3f})')
    ax.set_xlabel('Age Group')
    ax.set_ylabel('PR-AUC')
    ax.set_title('Precision-Recall Performance by Age Group')
    ax.legend()

    plt.tight_layout()
    plt.show()

### Performance by Gender

In [ ]:
if 'gender' in test_data.columns and test_data['gender'].notna().any():
    gender_metrics = calculate_subgroup_metrics(test_data, y_test, y_prob, 'gender')

    print("Performance by Gender:")
    print("=" * 80)
    print(gender_metrics.to_string(index=False))
else:
    gender_metrics = pd.DataFrame()
    print("Gender column not available - skipping gender analysis")


In [ ]:
if len(gender_metrics) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))

    x = np.arange(len(gender_metrics))
    width = 0.35

    bars1 = ax.bar(x - width/2, gender_metrics['auroc'], width, label='AUROC', color='steelblue')
    bars2 = ax.bar(x + width/2, gender_metrics['prauc'], width, label='PR-AUC', color='darkorange')

    ax.axhline(y=baseline_auroc, color='steelblue', linestyle='--', alpha=0.5)
    ax.axhline(y=baseline_prauc, color='darkorange', linestyle='--', alpha=0.5)

    ax.set_xlabel('Gender')
    ax.set_ylabel('Score')
    ax.set_title('Model Performance by Gender')
    ax.set_xticks(x)
    ax.set_xticklabels(gender_metrics['group'])
    ax.legend()

    plt.tight_layout()
    plt.show()

### Fairness Metrics

Beyond overall performance, fairness requires analyzing **decision-level metrics** for each group.

| Metric | Formula | Fairness Requirement |
|--------|---------|---------------------|
| **Equal Opportunity** | TPR | Same sensitivity across groups |
| **Predictive Parity** | PPV | Same precision across groups |
| **Equalized Odds** | TPR + FPR | Same error rates across groups |

---

<details>
<summary><strong>Hint 1 — Confusion matrix unpacking</strong> (click to expand)</summary>

```python
y_pred = (y_prob >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
tpr = tp / (tp + fn)  # sensitivity / recall
fpr = fp / (fp + tn)  # 1 - specificity
ppv = tp / (tp + fp)  # precision
```
</details>

In [ ]:
def calculate_fairness_metrics(y_true, y_prob, threshold=0.5):
    """
    Calculate classification fairness metrics at a given threshold.

    Args:
        y_true: true labels (0/1)
        y_prob: predicted probabilities
        threshold: classification threshold

    Returns:
        dict with tpr (sensitivity), fpr (1-specificity), ppv (precision)
    """
    # TODO: Compute binary predictions at the given threshold
    y_pred = None

    # TODO: Handle edge case (no positives or no negatives)
    if y_true.sum() == 0 or (y_true == 0).sum() == 0:
        return {'tpr': np.nan, 'fpr': np.nan, 'ppv': np.nan}

    # TODO: Compute confusion matrix values
    #   Use confusion_matrix(y_true, y_pred).ravel() -> tn, fp, fn, tp
    tn, fp, fn, tp = None, None, None, None

    # TODO: Compute TPR, FPR, PPV
    #   TPR = tp / (tp + fn)
    #   FPR = fp / (fp + tn)
    #   PPV = tp / (tp + fp)  (handle division by zero)
    tpr = None
    fpr = None
    ppv = None

    return {'tpr': tpr, 'fpr': fpr, 'ppv': ppv}

print("Fairness metrics function defined")

In [ ]:
# Calculate fairness metrics by age group
fairness_results = []

if 'age_group' in test_data.columns:
    for group in test_data['age_group'].dropna().unique():
        mask = test_data['age_group'] == group
        y_true_sub = y_test[mask]
        y_prob_sub = y_prob[mask]

        if len(y_true_sub) < 20 or y_true_sub.sum() < 5:
            continue

        metrics = calculate_fairness_metrics(y_true_sub, y_prob_sub, threshold=0.5)
        metrics['group'] = group
        metrics['n'] = len(y_true_sub)
        metrics['base_rate'] = y_true_sub.mean()
        fairness_results.append(metrics)

fairness_df = pd.DataFrame(fairness_results)

# Find the max-F1 threshold from the PR curve
from sklearn.metrics import precision_recall_curve
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precision_vals[:-1] * recall_vals[:-1] / (precision_vals[:-1] + recall_vals[:-1] + 1e-10)
max_f1_threshold = thresholds_pr[np.argmax(f1_scores)]
max_f1_score = f1_scores.max()

print(f"Default threshold: 0.5")
print(f"Max-F1 threshold:  {max_f1_threshold:.4f} (F1={max_f1_score:.3f})")
print(f"\nNote: With {y_test.mean():.1%} positive rate, 0.5 is far from optimal.")
print(f"The PR-curve-derived threshold better balances precision and recall.\n")

# Show fairness at BOTH thresholds
print("Fairness Metrics by Age Group (threshold=0.5):")
print("=" * 80)
print(fairness_df.to_string(index=False))

# Now at max-F1 threshold
fairness_maxf1 = []
if 'age_group' in test_data.columns:
    for group in test_data['age_group'].dropna().unique():
        mask = test_data['age_group'] == group
        y_true_sub = y_test[mask]
        y_prob_sub = y_prob[mask]
        if len(y_true_sub) < 20 or y_true_sub.sum() < 5:
            continue
        metrics = calculate_fairness_metrics(y_true_sub, y_prob_sub, threshold=max_f1_threshold)
        metrics['group'] = group
        metrics['n'] = len(y_true_sub)
        fairness_maxf1.append(metrics)

fairness_maxf1_df = pd.DataFrame(fairness_maxf1)
print(f"\nFairness Metrics by Age Group (threshold={max_f1_threshold:.4f}, max-F1):")
print("=" * 80)
print(fairness_maxf1_df.to_string(index=False))

# Compare TPR disparity
tpr_disp_05 = fairness_df['tpr'].max() - fairness_df['tpr'].min()
tpr_disp_f1 = fairness_maxf1_df['tpr'].max() - fairness_maxf1_df['tpr'].min()
print(f"\nTPR disparity at 0.5:   {tpr_disp_05:.2%}")
print(f"TPR disparity at max-F1: {tpr_disp_f1:.2%}")


In [ ]:
# Show fairness at BOTH thresholds side by side
if len(fairness_df) > 0 and len(fairness_maxf1_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    groups_str = fairness_df['group'].astype(str)
    x = np.arange(len(groups_str))
    width = 0.35

    # TPR (Sensitivity) — both thresholds
    ax = axes[0]
    ax.bar(x - width/2, fairness_df['tpr'], width, label='threshold=0.5', color='green', alpha=0.6)
    ax.bar(x + width/2, fairness_maxf1_df['tpr'], width, label=f'threshold={max_f1_threshold:.2f}', color='darkgreen', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(groups_str, rotation=45, ha='right')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Sensitivity by Age Group')
    ax.set_ylim([0, 1])
    ax.legend(fontsize=8)

    # FPR — both thresholds
    ax = axes[1]
    ax.bar(x - width/2, fairness_df['fpr'], width, label='threshold=0.5', color='salmon', alpha=0.6)
    ax.bar(x + width/2, fairness_maxf1_df['fpr'], width, label=f'threshold={max_f1_threshold:.2f}', color='darkred', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(groups_str, rotation=45, ha='right')
    ax.set_ylabel('False Positive Rate')
    ax.set_title('FPR by Age Group')
    ax.set_ylim([0, max(fairness_df['fpr'].max(), fairness_maxf1_df['fpr'].max()) * 1.3 + 0.01])
    ax.legend(fontsize=8)

    # PPV — both thresholds
    ax = axes[2]
    ax.bar(x - width/2, fairness_df['ppv'], width, label='threshold=0.5', color='cornflowerblue', alpha=0.6)
    ax.bar(x + width/2, fairness_maxf1_df['ppv'], width, label=f'threshold={max_f1_threshold:.2f}', color='darkblue', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(groups_str, rotation=45, ha='right')
    ax.set_ylabel('Positive Predictive Value')
    ax.set_title('PPV by Age Group')
    ax.set_ylim([0, 1])
    ax.legend(fontsize=8)

    plt.suptitle('Fairness Metrics: Default (0.5) vs Operating Threshold', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

### Disparity Analysis

In [ ]:
print("DISPARITY ANALYSIS")
print("=" * 60)

if len(age_metrics) > 0:
    max_auroc = age_metrics['auroc'].max()
    min_auroc = age_metrics['auroc'].min()
    auroc_disparity = max_auroc - min_auroc

    best_group = age_metrics.loc[age_metrics['auroc'].idxmax(), 'group']
    worst_group = age_metrics.loc[age_metrics['auroc'].idxmin(), 'group']

    print(f"\nAUROC Disparity (Age):")
    print(f"  Best:  {best_group} ({max_auroc:.4f})")
    print(f"  Worst: {worst_group} ({min_auroc:.4f})")
    print(f"  Gap:   {auroc_disparity:.4f}")
else:
    print("\nInsufficient data for age disparity analysis")


### Calibration Analysis

A model can rank patients well (high AUROC) but still output poorly calibrated probabilities. **Calibration** measures whether a predicted probability of 70% actually corresponds to a 70% event rate.

**Why calibration matters in healthcare:**
- Risk scores are used to prioritize interventions — if "80% risk" really means 30% risk, resources are misallocated
- Overconfident models create false urgency; underconfident models miss true emergencies
- Brier score measures overall calibration quality (lower = better)

**Common issue:** Models trained with `scale_pos_weight` or `class_weight='balanced'` tend to be **overconfident** — they output probabilities higher than the true event rate. Platt scaling (sigmoid recalibration) can fix this.

---

<details>
<summary><strong>Hint 1 — Calibration curve API</strong> (click to expand)</summary>

`prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy='uniform')` — then plot `prob_pred` on x-axis and `prob_true` on y-axis. A perfectly calibrated model follows the diagonal.
</details>

In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

# TODO: Compute Brier score
#   brier_score_loss(y_true, y_prob) — lower is better (0 = perfect)
brier = None
print(f"Brier Score: {brier}")

# TODO: Compute calibration curve
#   calibration_curve(y_test, y_prob, n_bins=10, strategy='uniform')
#   Returns: prob_true (actual fraction of positives), prob_pred (mean predicted probability)
prob_true, prob_pred = None, None

# TODO: Plot calibration curve
#   1. Create 1x2 subplot figure
#   2. Left: Plot predicted vs actual probability + diagonal reference line
#   3. Right: Plot histogram of predicted probabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Calibration plot
ax = axes[0]
# TODO: Plot prob_pred vs prob_true, add diagonal reference line
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve')

# Right panel: Prediction distribution
ax = axes[1]
# TODO: ax.hist(y_prob, bins=50, ...)
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Count')
ax.set_title('Prediction Distribution')

plt.tight_layout()
plt.show()

# TODO: Check for overconfidence
#   1. Compute mean predicted probability: y_prob.mean()
#   2. Compute actual event rate: y_test.mean()
#   3. If mean_pred > actual_rate * 2, model is overconfident
#   4. Print comparison
mean_pred = y_prob.mean()
actual_rate = y_test.mean()
print(f"\nMean predicted probability: {mean_pred:.4f}")
print(f"Actual event rate:         {actual_rate:.4f}")


---

# Part 2: Cross-Validation Stability

We use **GroupKFold** to assess model stability. Standard K-Fold would put the same patient's instances in both train and test, causing data leakage. GroupKFold ensures complete patient separation.

---

<details>
<summary><strong>Hint 1 — GroupKFold CV loop</strong> (click to expand)</summary>

```python
fold_model = xgb.XGBClassifier(n_estimators=model.n_estimators, max_depth=model.max_depth,
                                learning_rate=model.learning_rate, scale_pos_weight=spw,
                                random_state=42, eval_metric='aucpr')
fold_model.fit(X_tr, y_tr)
y_val_prob = fold_model.predict_proba(X_val)[:, 1]
```
</details>

In [ ]:
gkf = GroupKFold(n_splits=5)

print("Running 5-fold GroupKFold cross-validation...")

# TODO: Implement cross-validation loop
#   For each fold:
#     1. Split X, y using train/val indices
#     2. Compute scale_pos_weight from training labels
#     3. Create XGBClassifier with same params as loaded model
#     4. Fit on training fold
#     5. Evaluate on validation fold (AUROC + PRAUC)
#     6. Store results
cv_results = []

for fold, (train_idx_cv, val_idx_cv) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_val = X.iloc[train_idx_cv], X.iloc[val_idx_cv]
    y_tr, y_val = y.iloc[train_idx_cv], y.iloc[val_idx_cv]

    spw = (y_tr == 0).sum() / (y_tr == 1).sum()

    # TODO: Create and fit XGBClassifier
    fold_model = None

    # TODO: Predict and compute metrics
    y_val_prob = None
    fold_auroc = None
    fold_prauc = None

    cv_results.append({
        'fold': fold + 1,
        'train_size': len(X_tr),
        'val_size': len(X_val),
        'train_events': int(y_tr.sum()),
        'val_events': int(y_val.sum()),
        'auroc': fold_auroc,
        'prauc': fold_prauc,
    })

    print(f"  Fold {fold+1}: val={len(X_val):,}, events={int(y_val.sum()):,}, "
          f"AUROC={fold_auroc}, PRAUC={fold_prauc}")

cv_df = pd.DataFrame(cv_results)
print(f"\nMean AUROC: {cv_df['auroc'].mean()}")
print(f"Mean PRAUC: {cv_df['prauc'].mean()}")
print(f"Std AUROC:  {cv_df['auroc'].std()}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = cv_df['fold']
ax.plot(x, cv_df['auroc'], 'o-', label='AUROC', color='steelblue', markersize=10, linewidth=2)
ax.plot(x, cv_df['prauc'], 's-', label='PR-AUC', color='darkorange', markersize=10, linewidth=2)

ax.axhline(y=cv_df['auroc'].mean(), color='steelblue', linestyle='--', alpha=0.5)
ax.axhline(y=cv_df['prauc'].mean(), color='darkorange', linestyle='--', alpha=0.5)

ax.fill_between(x, cv_df['auroc'].mean() - cv_df['auroc'].std(),

    cv_df['auroc'].mean() + cv_df['auroc'].std(),

    alpha=0.2, color='steelblue')

ax.set_xlabel('Fold')
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Performance Stability')
ax.legend()
ax.set_xticks(x)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Temporal Validation

Cross-validation measures stability across random patient splits. **Temporal validation** tests whether the model generalizes across time — training on past data and predicting future data, which simulates real deployment.

**How the split works**: Each patient is assigned to train or test based on when they **first appear** in the dataset. The earliest ~80% of patients go to train, the latest ~20% go to test. All rows from a patient stay on the same side — no patient appears in both sets.

---

<details>
<summary><strong>Hint 1 — Temporal split approach</strong> (click to expand)</summary>

1. `patient_first_date.iloc[:cutoff_idx].index` gives train patient IDs
2. `patient_ids.isin(train_patients)` creates the row-level mask
3. Train XGBoost on temporal train set, evaluate on temporal test set
</details>

<details>
<summary><strong>Hint 2 — Why temporal validation matters</strong> (click to expand)</summary>

Random splits can place future data in training and past data in testing. Temporal validation ensures the model only sees past data during training — matching real-world deployment where you predict future events.
</details>

In [ ]:
# Temporal validation with strict patient isolation.
# TODO: Implement temporal train/test split
#   1. Parse dates, find each patient's earliest date
#   2. Sort patients by first appearance
#   3. Use earliest ~80% of patients for training, latest ~20% for testing
#   4. All rows from a patient go into the same split (no leakage)
#   5. Train XGBoost on temporal train, evaluate on temporal test

dates = pd.to_datetime(data_with_demo['date'])
patient_ids = data_with_demo[patient_col]

# TODO Step 1: Find each patient's earliest date and sort
patient_first_date = dates.groupby(patient_ids).min().sort_values()
n_patients = len(patient_first_date)
cutoff_idx = int(n_patients * 0.8)
temporal_cutoff = patient_first_date.iloc[cutoff_idx]

# TODO Step 2: Split patients into train (before cutoff) and test (after cutoff)
train_patients = None  # patients whose first date < cutoff
test_patients = None   # patients whose first date >= cutoff

# TODO Step 3: Create train/test masks for all rows
train_mask = None
test_mask = None

X_temp_train = X[train_mask]
y_temp_train = y[train_mask]
X_temp_test = X[test_mask]
y_temp_test = y[test_mask]

print(f"Temporal cutoff date: {temporal_cutoff.strftime('%Y-%m-%d')}")
print(f"Train patients: {len(train_patients):,}, Test patients: {len(test_patients):,}")
print(f"Train rows: {len(X_temp_train):,}, Test rows: {len(X_temp_test):,}")

# TODO Step 4: Train XGBoost on temporal train set
spw = (y_temp_train == 0).sum() / (y_temp_train == 1).sum()
temp_model = None  # TODO: Create and fit XGBClassifier

# TODO Step 5: Evaluate on temporal test set
y_temp_prob = None
temporal_auroc = None
temporal_prauc = None

print(f"\nTemporal Validation Results:")
print(f"  AUROC: {temporal_auroc}")
print(f"  PRAUC: {temporal_prauc}")

**Temporal Validation Interpretation**

This split is stricter than a naive date cutoff: patients who appear in the test period are completely removed from training. This prevents the model from memorizing patient-specific patterns during training and then "recognizing" the same patients in the test set at later dates.

Check the degradation: modest (~1-2% AUROC) is normal and confirms the model generalizes across time. Larger drops suggest temporal distribution drift that needs investigation.

---

# Part 3: Robustness Testing

We simulate missing data by **masking** features (setting them to NaN) and measuring performance degradation. Since XGBoost handles NaN natively, this tests how gracefully the model degrades when data sources become unavailable.

## 3.1 Feature Category Analysis

Features map to real-world data sources: lab systems, EHR encounter records, pharmacy systems, and care coordination workflows. If one system goes down, an entire category of features becomes unavailable.

In [ ]:
def categorize_feature(feature_name):
    """Categorize a feature based on its name."""
    name = feature_name.lower()

    if 'hba1c' in name or 'bmi' in name:
        return 'Lab/Clinical Values'
    elif 'egfr' in name:
        return 'Kidney Function'
    elif 'bp' in name or 'systolic' in name or 'diastolic' in name:
        return 'Blood Pressure'
    elif 'encounter' in name or 'emergency' in name or 'hospitalization' in name:
        return 'Encounter History'
    elif 'medication' in name:
        return 'Medication Data'
    elif 'complication' in name:
        return 'Clinical State'
    elif 'gap' in name:
        return 'Care Continuity'
    elif 'age' in name:
        return 'Demographics'
    else:
        return 'Other'

# Create category dataframe
category_df = pd.DataFrame({
    'feature': feature_cols,
    'category': [categorize_feature(f) for f in feature_cols]
})

print("Feature categories defined:")
print(category_df['category'].value_counts())

## 3.2 Category-Level Masking Test

Mask all features in each category and measure performance degradation. Categories with <90% retention are critical dependencies.

---

<details>
<summary><strong>Hint 1 — Feature masking</strong> (click to expand)</summary>

```python
X_masked = X_test.copy()
X_masked[col] = np.nan  # XGBoost handles NaN natively
y_prob_masked = model.predict_proba(X_masked)[:, 1]
```
</details>

In [ ]:
def evaluate_with_masked_features(X_test, y_test, model, features_to_mask, mask_value=np.nan):
    """
    Evaluate model performance when certain features are masked.

    Args:
        X_test: test feature DataFrame
        y_test: true labels
        model: trained model
        features_to_mask: list of feature names to mask
        mask_value: replacement value (default NaN — XGBoost handles natively)

    Returns:
        dict with auroc, prauc, n_masked
    """
    # TODO: Create a copy of X_test and set masked features to mask_value
    X_masked = X_test.copy()
    for col in features_to_mask:
        if col in X_masked.columns:
            X_masked[col] = mask_value

    # TODO: Predict with the model and compute metrics
    y_prob_masked = None
    auroc = None
    prauc = None

    return {
        'auroc': auroc,
        'prauc': prauc,
        'n_masked': len(features_to_mask),
    }

print("Feature masking evaluation function defined")

In [ ]:
# Run masking test for each feature category
category_results = []

for category in category_df['category'].unique():
    cat_features = category_df[category_df['category'] == category]['feature'].tolist()
    result = evaluate_with_masked_features(X_test, y_test, model, cat_features)
    result['category'] = category
    result['n_features_in_category'] = len(cat_features)
    result['auroc_retention'] = result['auroc'] / baseline_auroc
    result['prauc_retention'] = result['prauc'] / baseline_prauc
    category_results.append(result)

    status = 'PASS' if result['auroc_retention'] >= 0.9 else 'FAIL'
    print(f"  [{status}] {category}: {result['auroc_retention']:.1%} retention "
          f"({len(cat_features)} features masked)")

category_results_df = pd.DataFrame(category_results)
print(f"\nAll categories tested: {len(category_results_df)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

plot_data = category_results_df.sort_values('auroc_retention')
colors = ['red' if r < 0.9 else 'orange' if r < 0.95 else 'green'

    for r in plot_data['auroc_retention']]

bars = ax.barh(plot_data['category'], plot_data['auroc_retention'], color=colors, alpha=0.8)

ax.axvline(x=1.0, color='blue', linestyle='-', linewidth=2, label='Baseline')
ax.axvline(x=0.9, color='red', linestyle='--', linewidth=2, label='90% Threshold')

ax.set_xlabel('AUROC Retention (fraction of baseline)')
ax.set_title('Model Robustness: Impact of Missing Feature Categories')
ax.legend()
ax.set_xlim([0.5, 1.1])

for bar, val in zip(bars, plot_data['auroc_retention']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,

    f'{val:.1%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 3.3 Random Feature Masking

Simulate random data quality issues by masking X% of features randomly.

---

<details>
<summary><strong>Hint 1 — Random masking loop</strong> (click to expand)</summary>

```python
features_to_mask = list(np.random.choice(feature_cols, n_mask, replace=False))
result = evaluate_with_masked_features(X_test, y_test, model, features_to_mask)
```

Run 5 trials per percentage level and average the results.
</details>

In [ ]:
# TODO: Test model robustness by randomly masking increasing percentages of features
#   For each percentage (0%, 10%, 20%, ..., 90%):
#     1. Calculate how many features to mask: n_mask = int(len(feature_cols) * pct / 100)
#     2. Run 5 random trials (different random feature subsets each time)
#     3. Average the AUROC and PRAUC across trials
#     4. Store results

np.random.seed(42)
random_mask_results = []
mask_percentages = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

print("Testing random feature masking...")

for pct in mask_percentages:
    n_mask = int(len(feature_cols) * pct / 100)

    if n_mask == 0:
        random_mask_results.append({
            'pct_masked': pct,
            'auroc': baseline_auroc,
            'prauc': baseline_prauc
        })
    else:
        trial_aurocs = []
        trial_praucs = []

        for trial in range(5):
            # TODO: Randomly select n_mask features to mask
            features_to_mask = None  # np.random.choice(feature_cols, n_mask, replace=False)

            # TODO: Evaluate with masked features
            result = None  # evaluate_with_masked_features(...)

            trial_aurocs.append(None)  # result['auroc']
            trial_praucs.append(None)  # result['prauc']

        random_mask_results.append({
            'pct_masked': pct,
            'auroc': np.mean(trial_aurocs) if trial_aurocs[0] is not None else None,
            'prauc': np.mean(trial_praucs) if trial_praucs[0] is not None else None,
        })

    print(f"  {pct:2d}% masked: AUROC={random_mask_results[-1]['auroc']}, "
          f"PRAUC={random_mask_results[-1]['prauc']}")

random_mask_df = pd.DataFrame(random_mask_results)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(random_mask_df['pct_masked'], random_mask_df['auroc'],

    'o-', color='steelblue', linewidth=2, markersize=10, label='AUROC')

if 'auroc_std' in random_mask_df.columns:
    ax.fill_between(random_mask_df['pct_masked'],
    random_mask_df['auroc'] - random_mask_df['auroc_std'].fillna(0),
    random_mask_df['auroc'] + random_mask_df['auroc_std'].fillna(0),
    alpha=0.2, color='steelblue')

ax.axhline(y=baseline_auroc * 0.9, color='red', linestyle='--',

    label=f'90% of Baseline ({baseline_auroc*0.9:.3f})')
ax.axhline(y=0.5, color='gray', linestyle=':', label='Random Guessing')

ax.set_xlabel('Percentage of Features Masked (%)')
ax.set_ylabel('AUROC')
ax.set_title('Model Degradation with Random Feature Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([-5, 95])
ax.set_ylim([0.4, 1.0])

plt.tight_layout()
plt.show()

**Interpreting the curve:** If the curve is non-monotonic (AUROC increases at some masking levels), this is expected with only 5 random trials over 22 features. Different random subsets can accidentally remove noisy or redundant features that hurt more than they help. The overall trend should be downward, but individual points may jitter. For a smoother curve, increase the trial count.

## 3.4 Single Feature Removal

Test the impact of removing each top-10 feature individually to identify single points of failure.

In [ ]:
importance_dict = dict(zip(feature_cols, model.feature_importances_))
importance_sorted = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

print("Impact of removing single features (top 10):")
print("=" * 70)

single_feature_results = []

for feat, imp in importance_sorted[:10]:
    result = evaluate_with_masked_features(X_test, y_test, model, [feat])
    retention = result['auroc'] / baseline_auroc


    single_feature_results.append({
    'feature': feat,
    'importance': imp,
    'auroc_masked': result['auroc'],
    'auroc_retention': retention
    })


    status = 'FAIL' if retention < 0.9 else 'PASS'
    print(f"  [{status}] Without '{feat[:40]}': {result['auroc']:.4f} ({retention:.1%})")


---

# Part 4: Error Analysis

Understanding **why** the model makes mistakes is crucial for clinical deployment.

## 4.1 Multivariate Outlier Detection

Per-feature outlier detection (flagging any value outside the 5th-95th percentile) is too aggressive — it flags 53%+ of data as "outlier" because with 22 features, most patients are extreme on at least one.

Instead, we use **Isolation Forest**, which detects outliers in the joint feature space. It identifies patients whose overall feature profile is unusual, not just those with one extreme value. With `contamination=0.05`, it flags approximately 5% of data — a much more actionable set for clinical review.

---

<details>
<summary><strong>Hint 1 — Isolation Forest API</strong> (click to expand)</summary>

```python
iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
labels = iso.fit_predict(X_filled)  # returns -1 for outliers, 1 for normal
outlier_mask = labels == -1
```
</details>

In [ ]:
from sklearn.ensemble import IsolationForest

def identify_outliers(X, contamination=0.05):
    """
    Identify multivariate outliers using Isolation Forest.

    Args:
        X: feature DataFrame
        contamination: expected fraction of outliers (0.05 = 5%)

    Returns:
        boolean array — True for outlier rows
    """
    # TODO: Create IsolationForest with contamination and random_state=42
    # TODO: Fill NaN (IsolationForest can't handle missing values)
    # TODO: fit_predict returns -1 for outliers, 1 for normal
    #   HINT: iso.fit_predict(X_filled) == -1
    iso = None
    X_filled = X.fillna(X.median())
    outlier_labels = None  # iso.fit_predict(X_filled)

    return outlier_labels == -1

# TODO: Apply to test set and compare outlier vs normal performance
outlier_mask = identify_outliers(X_test, contamination=0.05)

print("OUTLIER IDENTIFICATION (Isolation Forest)")
print("=" * 50)
print(f"Total test instances: {len(X_test):,}")
print(f"Outliers (5%): {outlier_mask.sum():,}")
print(f"Normal: {(~outlier_mask).sum():,}")

In [ ]:
# Compare outlier vs normal performance
y_test_outlier = y_test[outlier_mask]
y_prob_outlier = y_prob[outlier_mask]
y_test_normal = y_test[~outlier_mask]
y_prob_normal = y_prob[~outlier_mask]

print("Performance Comparison:")
print("=" * 50)
print(f"\nBaseline (all data): AUROC = {baseline_auroc:.4f}")

if len(y_test_normal) > 10 and y_test_normal.sum() > 1:
    normal_auroc = roc_auc_score(y_test_normal, y_prob_normal)
    print(f"Normal patients (n={len(y_test_normal):,}): AUROC = {normal_auroc:.4f}")

if len(y_test_outlier) > 10 and y_test_outlier.sum() > 1:
    outlier_auroc = roc_auc_score(y_test_outlier, y_prob_outlier)
    print(f"Outlier patients (n={len(y_test_outlier):,}): AUROC = {outlier_auroc:.4f}")

## 4.2 False Negative Deep Dive

False negatives are **dangerous** in healthcare — these are high-risk patients we failed to identify.

In [ ]:
# Label prediction outcomes using max-F1 threshold (not 0.5)
test_data = test_data.copy()
test_data['y_true'] = y_test.values
test_data['y_prob'] = y_prob
test_data['y_pred'] = (y_prob >= max_f1_threshold).astype(int)

test_data['outcome'] = 'TN'
test_data.loc[(test_data['y_true'] == 1) & (test_data['y_pred'] == 1), 'outcome'] = 'TP'
test_data.loc[(test_data['y_true'] == 1) & (test_data['y_pred'] == 0), 'outcome'] = 'FN'
test_data.loc[(test_data['y_true'] == 0) & (test_data['y_pred'] == 1), 'outcome'] = 'FP'

fn_data = test_data[test_data['outcome'] == 'FN'].copy()
tp_data = test_data[test_data['outcome'] == 'TP'].copy()
fp_data = test_data[test_data['outcome'] == 'FP'].copy()
tn_data = test_data[test_data['outcome'] == 'TN'].copy()

print(f"Using max-F1 threshold: {max_f1_threshold:.4f}")
print(f"\nError Type Breakdown:")
print(f" True Positives (TP): {len(tp_data):,} - Correctly identified high-risk")
print(f" True Negatives (TN): {len(tn_data):,} - Correctly identified low-risk")
print(f" False Negatives (FN): {len(fn_data):,} - Missed high-risk patients")
print(f" False Positives (FP): {len(fp_data):,} - Unnecessary alarms")

In [ ]:
if len(fn_data) > 0 and len(tp_data) > 0:
    print("Comparing False Negatives to True Positives:")
    print("-" * 60)
    print(f"{'Feature':<40} {'FN Mean':>12} {'TP Mean':>12} {'Diff':>8}")
    print("-" * 60)

    for feat in feature_cols[:12]:
        fn_mean = fn_data[feat].mean()
        tp_mean = tp_data[feat].mean()
        diff = fn_mean - tp_mean

        indicator = '↓' if diff < -0.1 else '↑' if diff > 0.1 else ' '
        print(f"{feat[:39]:<40} {fn_mean:>12.2f} {tp_mean:>12.2f} {diff:>+7.2f} {indicator}")
else:
    print("Insufficient data for FN/TP comparison")

In [ ]:
if len(fn_data) > 0:
    print("\nFalse Negative Risk Score Distribution:")
    print(f" Mean predicted probability: {fn_data['y_prob'].mean():.1%}")
    print(f" Median: {fn_data['y_prob'].median():.1%}")
    print(f" Min: {fn_data['y_prob'].min():.1%}")
    print(f" Max: {fn_data['y_prob'].max():.1%}")

    near_threshold = ((fn_data['y_prob'] >= max_f1_threshold * 0.8) & (fn_data['y_prob'] < max_f1_threshold)).sum()
    print(f"\n Near threshold ({max_f1_threshold*0.8:.2f}-{max_f1_threshold:.2f}): {near_threshold} ({near_threshold/len(fn_data):.1%})")
    print(" → These patients could be caught with a slightly lower threshold")

## 4.3 False Positive Deep Dive

False positives cause **alert fatigue** — clinicians stop trusting alerts if too many are wrong.

In [ ]:
if len(fp_data) > 0 and len(tn_data) > 0:
    print("Comparing False Positives to True Negatives:")
    print("-" * 60)
    print(f"{'Feature':<40} {'FP Mean':>12} {'TN Mean':>12} {'Diff':>8}")
    print("-" * 60)

    for feat in feature_cols[:12]:
        fp_mean = fp_data[feat].mean()
        tn_mean = tn_data[feat].mean()
        diff = fp_mean - tn_mean

        indicator = '↑' if diff > 0.1 else '↓' if diff < -0.1 else ' '
        print(f"{feat[:39]:<40} {fp_mean:>12.2f} {tn_mean:>12.2f} {diff:>+7.2f} {indicator}")
else:
    print("Insufficient data for FP/TN comparison")

In [ ]:
if len(fp_data) > 0:
    print("\nFalse Positive Risk Score Distribution:")
    print(f" Mean predicted probability: {fp_data['y_prob'].mean():.1%}")
    print(f" Median: {fp_data['y_prob'].median():.1%}")


    high_conf_fp = (fp_data['y_prob'] >= 0.8).sum()
    print(f"\n High confidence FPs (>80%): {high_conf_fp} ({high_conf_fp/len(fp_data):.1%})")
    print(" → These are concerning - model is confidently wrong")

---

# Part 5: Combined Assessment Report

Compile all findings into a structured report for technical review and EU AI Act documentation.

In [ ]:
# Compute summary metrics for the report
n_fn = len(fn_data)
n_fp = len(fp_data)
n_tp = len(tp_data)
n_tn = len(tn_data)

sensitivity = n_tp / (n_tp + n_fn) if (n_tp + n_fn) > 0 else 0
specificity = n_tn / (n_tn + n_fp) if (n_tn + n_fp) > 0 else 0
ppv = n_tp / (n_tp + n_fp) if (n_tp + n_fp) > 0 else 0

tpr_disparity_05 = fairness_df['tpr'].max() - fairness_df['tpr'].min() if len(fairness_df) > 0 else 0
tpr_disparity_maxf1 = fairness_maxf1_df['tpr'].max() - fairness_maxf1_df['tpr'].min() if len(fairness_maxf1_df) > 0 else 0
auroc_disparity = age_metrics['auroc'].max() - age_metrics['auroc'].min() if len(age_metrics) > 0 else 0
categories_passing = (category_results_df['auroc_retention'] >= 0.9).sum()

# Worst subgroup at operating threshold
if len(fairness_maxf1_df) > 0:
    worst_tpr_group = fairness_maxf1_df.loc[fairness_maxf1_df['tpr'].idxmin()]
    print(f"WORST SUBGROUP at operating threshold ({max_f1_threshold:.4f}):")
    print(f"  {worst_tpr_group['group']}: TPR = {worst_tpr_group['tpr']:.1%}")

print("\nREPORT VALUES")
print(f"  Baseline AUROC: {baseline_auroc:.4f}, PR-AUC: {baseline_prauc:.4f}")
print(f"  Brier Score: {brier:.4f}")
print(f"  Max-F1 threshold: {max_f1_threshold:.4f} (F1={max_f1_score:.3f})")
print(f"  Mean predicted prob: {y_prob.mean():.4f}, Actual rate: {y_test.mean():.4f}")
print(f"  AUROC disparity (age): {auroc_disparity:.4f}")
print(f"  TPR disparity at 0.5:       {tpr_disparity_05:.2%}")
print(f"  TPR disparity at max-F1:    {tpr_disparity_maxf1:.2%}")
print(f"  CV AUROC: {cv_df['auroc'].mean():.4f} +/- {cv_df['auroc'].std():.4f}")
print(f"  Temporal AUROC: {temporal_auroc:.4f}")
print(f"  Categories passing 90%: {categories_passing}/{len(category_results_df)}")
print(f"  TP={n_tp}, TN={n_tn}, FP={n_fp}, FN={n_fn}")
print(f"  Sensitivity: {sensitivity:.1%}, Specificity: {specificity:.1%}, PPV: {ppv:.1%}")

## Model Assessment Report

> **Note:** The values below are from the code cell above. Re-run the notebook if they look stale.

### 1. Baseline Performance

| Metric | Value |
|--------|-------|
| AUROC | *(see cell output)* |
| PR-AUC | *(see cell output)* |
| Brier Score | *(see cell output)* |
| Max-F1 Threshold | *(see cell output)* |

### 2. Calibration

Check the calibration output above: if the mean predicted probability is significantly higher than the actual event rate, the model is overconfident. Platt scaling (`CalibratedClassifierCV`) is recommended before deployment.

### 3. Fairness

| Metric | Threshold=0.5 | Operating Threshold (max-F1) |
|--------|---------------|------------------------------|
| TPR disparity | *(see cell output)* | *(see cell output)* |
| Worst subgroup | — | *(see cell output — check 18-40 group)* |

**The operating threshold matters more than 0.5.** If the 18-40 group's sensitivity drops significantly at the operating threshold, younger diabetic patients will be systematically under-detected. This must be addressed before deployment.

### 4. Temporal Validation (patient-isolated)

Train on earlier patients (by first appearance date, ~80%), test on later patients (~20%). Patient isolation prevents the same patient from leaking across the split. The exact cutoff date depends on the dataset.

### 5. Feature Category Robustness

Most categories meet 90% AUROC retention. Lab/Clinical Values are the critical dependency — need fallback logic when lab data is unavailable.

### 6. Random Feature Masking

With only 5 random trials per masking level and 22 features, the curve may be **non-monotonic** (e.g., AUROC increasing at high masking percentages). This is an artifact of small sample size: different random feature subsets can remove harmful noise. Do not interpret upward ticks as "masking helps" — increase trial count for smoother curves.

### 7. Error Analysis

Error analysis (FN/FP deep dive) uses the max-F1 threshold. For deeper investigation of *why* specific patients are misclassified, combine this with the SHAP explanations from Session 3 — generate per-patient SHAP force plots for false negatives to identify which features the model relied on incorrectly.

### 8. Deployment Recommendations

- Use max-F1 threshold instead of 0.5 — optimized for the ~2.9% positive rate
- Apply Platt scaling to fix overconfident probabilities
- **Address TPR disparity at the operating threshold** — the 0.5-threshold disparity understates the real problem
- Implement fallback logic when Lab/Clinical Values are unavailable
- Flag low-confidence predictions for human review
- Implement temporal monitoring to detect distribution drift

## Save All Artifacts

In [ ]:
# Save generalization results
if len(age_metrics) > 0:
    age_metrics.to_csv(os.path.join(OUTPUT_DIR, 'subgroup_performance_age.csv'), index=False)
    print(f"Saved: {os.path.join(OUTPUT_DIR, 'subgroup_performance_age.csv')}")

if len(gender_metrics) > 0:
    gender_metrics.to_csv(os.path.join(OUTPUT_DIR, 'subgroup_performance_gender.csv'), index=False)
    print(f"Saved: {os.path.join(OUTPUT_DIR, 'subgroup_performance_gender.csv')}")

cv_df.to_csv(os.path.join(OUTPUT_DIR, 'cross_validation_results.csv'), index=False)
print(f"Saved: {os.path.join(OUTPUT_DIR, 'cross_validation_results.csv')}")

if len(fairness_df) > 0:
    fairness_df.to_csv(os.path.join(OUTPUT_DIR, 'fairness_metrics.csv'), index=False)
    print(f"Saved: {os.path.join(OUTPUT_DIR, 'fairness_metrics.csv')}")

# Save robustness results
category_results_df.to_csv(os.path.join(OUTPUT_DIR, 'robustness_category_impact.csv'), index=False)
print(f"Saved: {os.path.join(OUTPUT_DIR, 'robustness_category_impact.csv')}")

random_mask_df.to_csv(os.path.join(OUTPUT_DIR, 'robustness_random_masking.csv'), index=False)
print(f"Saved: {os.path.join(OUTPUT_DIR, 'robustness_random_masking.csv')}")

error_summary = pd.DataFrame([{
    'metric': 'false_negatives', 'count': n_fn, 'rate': np.nan
}, {
    'metric': 'false_positives', 'count': n_fp, 'rate': np.nan
}, {
    'metric': 'sensitivity', 'count': np.nan, 'rate': sensitivity
}, {
    'metric': 'specificity', 'count': np.nan, 'rate': specificity
}])
error_summary.to_csv(os.path.join(OUTPUT_DIR, 'robustness_error_analysis.csv'), index=False)
print(f"Saved: {os.path.join(OUTPUT_DIR, 'robustness_error_analysis.csv')}")


---

## Summary

### What You Accomplished

1. **Generalization**: Evaluated model across age and gender subgroups using per-instance age (not a fixed reference date)
2. **Fairness at two thresholds**: Computed TPR disparity at both 0.5 and the operating max-F1 threshold — the operating threshold reveals larger disparities
3. **Calibration**: Assessed probability calibration with Brier score and calibration curves
4. **Stability**: 5-fold GroupKFold cross-validation confirmed consistent performance
5. **Temporal Validation**: Patient-isolated temporal split (patients in test period excluded from training entirely)
6. **Robustness**: Tested feature category masking, random feature loss, and single feature removal
7. **Error Analysis**: Identified outliers with Isolation Forest (~5%), deep-dived false negatives and positives
8. **Report**: Generated comprehensive assessment for EU AI Act compliance

### Key Findings

| Analysis | Finding | Implication |
|----------|---------|-------------|
| Subgroup AUROC | Varies across age groups | Check disparity at the operating threshold, not just ranking metrics |
| TPR at operating threshold | Worst subgroup may have much lower sensitivity | Younger patients potentially under-detected |
| CV Stability | AUROC std < 0.005 | Model performance is stable across folds |
| Temporal Validation | Patient-isolated split | Stricter than naive date cutoff; check degradation |
| Calibration | Model is overconfident | Apply Platt scaling before deployment |
| Category Robustness | Lab values critical (<90% retention) | Need fallback when lab data unavailable |
| Random Masking | Curve may be non-monotonic | Artifact of 5 trials; overall trend is downward |
| Outlier Detection | ~5% flagged by Isolation Forest | Multivariate approach avoids 53% false alarm rate |

### Next Steps

In the **homework**, you will apply all techniques from this week to create a comprehensive model assessment package.

### Professional Tip

Fairness and robustness are not one-time checks. Implement **ongoing monitoring** to detect if disparities emerge over time as patient populations shift. The EU AI Act requires documentation of subgroup performance before deployment AND continuous post-deployment monitoring.